# Measure Sandbox Widget

This notebook provides an interactive widget to:
1. Select a measure
2. Select one or more dimensions
3. Return a tabular DAX result
4. Control the order of returned columns
5. Temporarily edit the measure expression and format string
6. Revert or write changes back to the model

Draft edits are kept in the widget until you choose **Write Back To Model**.

In [3]:
# Install required packages for this Fabric notebook
%pip install -q semantic-link-labs semantic-link-sempy anywidget traitlets pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: C:\Users\edwar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
# Configure target semantic model
DATASET = "<your semantic model name>"
WORKSPACE = None  # or "Your Workspace"

In [5]:
import json
import re
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
from IPython.display import display

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "sempy_labs").exists()),
    None,
 )
if repo_root is not None:
    src_path = str(repo_root / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

import anywidget
import traitlets

from sempy_labs.tom import connect_semantic_model
from sempy_labs._dax import evaluate_dax_impersonation
from sempy_labs._helper_functions import format_dax_object_name

log_telemetry: sempy.relationships
log_telemetry: sempy.dependencies
log_telemetry: sempy.fabric
log_telemetry: sempy


In [ ]:
MEASURE_REF_PATTERN = re.compile(r"^'(?P<table>.+)'\[(?P<measure>.+)\]$")


def parse_measure_ref(measure_ref: str) -> Tuple[str, str]:
    match = MEASURE_REF_PATTERN.match(measure_ref)
    if not match:
        raise ValueError(f"Invalid measure reference: {measure_ref}")
    return match.group("table"), match.group("measure")


def load_model_metadata(dataset: str, workspace: Optional[str] = None) -> Dict[str, Dict]:
    measure_expr_by_ref: Dict[str, str] = {}
    measure_format_string_by_ref: Dict[str, str] = {}
    dimension_refs: List[str] = []

    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for measure in tom.all_measures():
            if getattr(measure, "IsHidden", False):
                continue
            table_name = measure.Parent.Name
            ref = format_dax_object_name(table=table_name, column=measure.Name)
            measure_expr_by_ref[ref] = measure.Expression or ""
            measure_format_string_by_ref[ref] = (getattr(measure, "FormatString", None) or "")

        for column in tom.all_columns():
            if getattr(column, "IsHidden", False):
                continue
            table_name = column.Parent.Name
            ref = format_dax_object_name(table=table_name, column=column.Name)
            dimension_refs.append(ref)

    return {
        "measure_expr_by_ref": dict(sorted(measure_expr_by_ref.items())),
        "measure_format_string_by_ref": dict(sorted(measure_format_string_by_ref.items())),
        "dimension_refs": sorted(set(dimension_refs)),
    }


def build_dax_query(measure_ref: str, dimension_refs: List[str], topn: int = 200) -> str:
    if not dimension_refs:
        return f"EVALUATE ROW(\"Value\", {measure_ref})"

    dim_block = ",\n        ".join(dimension_refs)
    return f"""
EVALUATE
TOPN(
    {topn},
    SUMMARIZECOLUMNS(
        {dim_block},
        \"Value\", {measure_ref}
    ),
    [Value], DESC
)
""".strip()


def preview_measure_table(dataset: str, workspace: Optional[str], measure_ref: str, dimension_refs: List[str], topn: int) -> pd.DataFrame:
    dax_query = build_dax_query(measure_ref=measure_ref, dimension_refs=dimension_refs, topn=topn)
    return evaluate_dax_impersonation(
        dataset=dataset,
        dax_query=dax_query,
        workspace=workspace,
    )


def update_measure_definition(
    dataset: str,
    workspace: Optional[str],
    measure_ref: str,
    expression: str,
    format_string: Optional[str] = None,
) -> None:
    table_name, measure_name = parse_measure_ref(measure_ref)
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=False) as tom:
        # Prefer the TOMWrapper API first; fall back to direct mutation for older wrappers.
        if hasattr(tom, "update_measure"):
            update_kwargs = {"measure_name": measure_ref, "expression": expression}
            if format_string is not None:
                update_kwargs["format_string"] = format_string
            try:
                tom.update_measure(**update_kwargs)
                return
            except TypeError:
                tom.update_measure(measure_name=measure_ref, expression=expression)
                if format_string is None:
                    return

        measure = tom.model.Tables[table_name].Measures[measure_name]
        measure.Expression = expression
        if format_string is not None:
            measure.FormatString = format_string